In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from yellowbrick.cluster import KElbowVisualizer
import matplotlib.pyplot as plt
import pandas as pd 
import seaborn as sns
from sklearn.cluster import KMeans
from sksurv.base import SurvivalAnalysisMixin as s
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sksurv.preprocessing import encode_categorical
from sksurv.datasets import load_gbsg2
from sksurv.functions import StepFunction
from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis
from sksurv.ensemble import (ComponentwiseGradientBoostingSurvivalAnalysis, 
                            RandomSurvivalForest, 
                            ExtraSurvivalTrees, 
                            GradientBoostingSurvivalAnalysis, 
                            ExtraSurvivalTrees)
from sksurv.meta import EnsembleSelection, EnsembleSelectionRegressor
from sksurv.metrics import integrated_brier_score
from matplotlib.colors import ListedColormap
from mlxtend.evaluate import paired_ttest_5x2cv
from mlxtend.evaluate import combined_ftest_5x2cv
from lifelines import KaplanMeierFitter
from scipy.cluster import hierarchy
from lifelines.statistics import logrank_test, multivariate_logrank_test, pairwise_logrank_test
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold, KFold
from lifelines.plotting import add_at_risk_counts
import scipy.stats
import sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import optuna
from sklearn.model_selection import cross_val_score
from sksurv.metrics import integrated_brier_score
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test
import scipy.stats as stats
from statsmodels.stats.outliers_influence import variance_inflation_factor 
from sklearn.preprocessing import StandardScaler
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

In [2]:
# OUS: Train data
OUS_D1 = pd.read_csv('OUS_D1.csv')
OUS_D2 = pd.read_csv('OUS_D2.csv')
OUS_D3 = pd.read_csv('OUS_D3.csv')
OUS_DFS_target = pd.read_csv('OUS_DFS_target.csv')
OUS_OS_target = pd.read_csv('OUS_OS_target.csv')
response_OUS = pd.read_csv('response_ous.csv', sep=';')

# MAASTRO: Test data 
MAASTRO_D1 = pd.read_csv('MAASTRO_D1.csv')
MAASTRO_D2 = pd.read_csv('MAASTRO_D2.csv')
MAASTRO_D3 = pd.read_csv('MAASTRO_D3.csv')
MAASTRO_DFS_target = pd.read_csv('MAASTRO_DFS_target.csv')
MAASTRO_OS_target = pd.read_csv('MAASTRO_OS_target.csv')
response_MAASTRO = pd.read_csv('maastro_response_full.csv', sep=',')

In [3]:
# Need to choose patient_id from OUS_D1 in response_OUS
data = list(OUS_D1['patient_id'])
mask = response_OUS['patient_id'].isin(data)
response_OUS = response_OUS[mask] 

# Merge OUS_D2 with response_OUS
clinical_train = pd.merge(OUS_D1, response_OUS, on='patient_id', how='inner')
clinical_train = clinical_train.loc[:, ~clinical_train.columns.isin(['OS', 'event_OS', 'LRC', 'event_LRC'])]

In [4]:
# Drop patient_id column
clinical_train = clinical_train.drop('patient_id', axis=1)

In [5]:
# Check null values in D1
clinical_train.isnull().sum().sum()

0

In [6]:
""" Takes too long time 
# Check collinearity for OUS data 
df = clinical_train

# Create a correlation matrix
correlation_matrix = df.corr()

plt.figure(figsize=(15, 12))

# Plot a heatmap
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=0.5)
plt.show()
""" 

" Takes too long time \n# Check collinearity for OUS data \ndf = clinical_train\n\n# Create a correlation matrix\ncorrelation_matrix = df.corr()\n\nplt.figure(figsize=(15, 12))\n\n# Plot a heatmap\nsns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=0.5)\nplt.show()\n"

## Test dataset: MAASTRO 

In [7]:
(MAASTRO_D1['patient_id'] == MAASTRO_OS_target['patient_id']).sum()

99

In [8]:
# Rename the column name of response_MAASTRO 
response_MAASTRO.rename(columns = {'Index' : 'patient_id'}, inplace = True)

In [9]:
# need to choose patient_id from MAASTRO_D2 in response_MAASTRO
data = list(MAASTRO_D1['patient_id'])
mask = response_MAASTRO['patient_id'].isin(data)
response_MAASTRO = response_MAASTRO[mask] 
response_MAASTRO

,patient_id,OS,OS_event,LRC,LRC_event,DFS,DFS_event
0,1,62.43,0.0,62.43,0.0,62.43,0.0
1,2,60.00,0.0,60.00,0.0,60.00,0.0
2,3,44.43,1.0,8.83,1.0,8.83,1.0
3,4,37.20,1.0,19.37,0.0,19.73,1.0
5,6,59.23,0.0,59.23,0.0,59.23,0.0
...,...,...,...,...,...,...,...
109,110,19.00,1.0,13.27,1.0,13.27,1.0
110,111,85.87,1.0,82.83,0.0,85.87,1.0
111,112,42.87,0.0,42.87,0.0,42.87,0.0
112,113,58.93,0.0,58.93,0.0,58.93,0.0


In [10]:
# Merge MAASTRO_D1 with response_MAASTRO
clinical_test = pd.merge(MAASTRO_D1, response_MAASTRO, on='patient_id', how='inner')
clinical_test = clinical_test.loc[:, ~clinical_test.columns.isin(['OS', 'OS_event', 'LRC', 'LRC_event'])]
clinical_test

,patient_id,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG,DFS,DFS_event
0,1,55,0,0,1,0,0,1,1,1,0,0,15.438583,22.841,263.611623,62.43,0.0
1,2,55,0,0,1,0,0,0,0,0,20,1,8.829353,5.660,36.980700,60.00,0.0
2,3,55,0,0,1,0,0,0,0,1,6,1,13.476123,7.791,74.636342,8.83,1.0
3,4,61,1,0,0,0,1,1,0,1,45,1,8.632732,7.908,46.791979,19.73,1.0
4,6,70,0,0,1,0,0,1,1,1,59,0,9.783954,15.237,107.637514,59.23,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,110,66,1,0,0,0,1,0,0,0,55,1,31.338410,6.110,144.490782,13.27,1.0
95,111,63,0,0,0,0,1,0,0,1,174,1,13.041604,7.182,69.214868,85.87,1.0
96,112,63,0,0,1,0,0,1,1,1,0,1,8.944517,16.483,102.594274,42.87,0.0
97,113,54,0,0,1,0,0,1,1,0,0,0,14.184236,9.981,103.229492,58.93,0.0


In [11]:
# Drop patient_id column
clinical_test = clinical_test.drop('patient_id', axis=1)

In [12]:
# Some rows have null values in OS, OS_event -> Remove those rows
clinical_test[clinical_test.isnull().any(axis=1)]
clinical_test = clinical_test.dropna(how='any',axis=0) 

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG,DFS,DFS_event


In [13]:
# X
X = clinical_train.loc[:, ~clinical_train.columns.isin(['DFS', 'event_DFS'])]

# y 
y = clinical_train.loc[:, ['DFS', 'event_DFS']]

# Set lower, upper time point and times for IBS calculation later 
lower, upper = np.percentile(y['DFS'], [10, 90])
times = np.arange(lower, upper)

In [14]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [15]:
# Shape
print('X_train: ', X.shape)
print('y_train: ', y.shape)

X_train:  (139, 14)
y_train:  (139,)


In [16]:
clinical_test.rename(columns = {'DFS_event' : 'event_DFS'}, inplace = True)

In [17]:
# X
X_MAASTRO = clinical_test.loc[:, ~clinical_test.columns.isin(['DFS', 'event_DFS'])]

# y y_MAASTRO
y_MAASTRO = clinical_test.loc[:, ['DFS', 'event_DFS']]
lower, upper = np.percentile(y_MAASTRO['DFS'], [10, 90])
times = np.arange(lower, upper)

# y into array 
lists = [] 
for i, j in zip(y_MAASTRO['event_DFS'], y_MAASTRO['DFS']): 
    lists.append((i, j))

y_MAASTRO = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

clinical_test.shape

(99, 16)

# Feature Selection: RENT

In [18]:
# Choose features from the result of RENT 
selected_features = ["hpv_related",
"pack_years",
"uicc8_III-IV",
"oropharynx",
"cavum_oris",
"TLG"]

In [19]:
X_rent = X.loc[:, selected_features]
X_new = X_rent.copy()

In [20]:
# Selecct the columns from X_MAASTRO
MAASTRO_new = X_MAASTRO.loc[:, selected_features]

# Standardization

In [21]:
# Copy the original X for later 
original_X = X.copy()

In [22]:
categorical_columns = ['female', 
                        'cavum_oris',
                        'oropharynx',
                        'hypopharynx',
                        'larynx',
                        'histgrade_high',
                        'hpv_related',
                        'charlson',
                        'uicc8_III-IV']

# Standardize X_new, the new data with the selected features only 
# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in X_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
X_new_categoric = X_new[columns_to_drop]
X_new_numeric = X_new.drop(columns=columns_to_drop, inplace=False)

# Do the standardization for the numeric part 
scaler = StandardScaler() 
X_new_numeric_columns = X_new_numeric.columns
X_new_numeric_index = X_new_numeric.index 
X_new_numeric_std = scaler.fit_transform(X_new_numeric)
X_new_numeric_std = pd.DataFrame(X_new_numeric_std,
                                 columns=X_new_numeric_columns, 
                                 index=X_new_numeric_index)
X_new_std = pd.concat([X_new_numeric_std, X_new_categoric], axis=1)

# Change the order of the X_new_std 
X_new_std = X_new_std[X_new.columns]

In [43]:
# Standardize X_MAASTRO 
# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in MAASTRO_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
MAASTRO_new_categoric = MAASTRO_new[columns_to_drop]
MAASTRO_new_numeric = MAASTRO_new.drop(columns=columns_to_drop, inplace=False)

# Do the standardization for the numeric part 
MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns
MAASTRO_new_numeric_index = MAASTRO_new_numeric.index 
MAASTRO_new_numeric_std = scaler.transform(MAASTRO_new_numeric)
MAASTRO_new_numeric_std = pd.DataFrame(MAASTRO_new_numeric_std,
                                 columns=MAASTRO_new_numeric_columns, 
                                 index=MAASTRO_new_numeric_index)
MAASTRO_new_std = pd.concat([MAASTRO_new_numeric_std, MAASTRO_new_categoric], axis=1)

# Change the order  
MAASTRO_new = MAASTRO_new[X_new.columns]
MAASTRO_new_std = MAASTRO_new_std[X_new.columns]

In [44]:
X_new

,hpv_related,pack_years,uicc8_III-IV,oropharynx,cavum_oris,TLG
0,0.0,0.000000,0.0,1,0,86.228420
1,0.0,27.404795,0.0,0,0,7.040100
2,0.0,41.019178,1.0,0,1,83.569669
3,0.0,37.500000,0.0,0,0,5.567091
4,0.0,53.000000,0.0,0,0,16.150550
...,...,...,...,...,...,...
134,1.0,0.000000,0.0,1,0,26.280140
135,1.0,0.000000,1.0,1,0,101.754834
136,1.0,39.498630,0.0,1,0,66.273201
137,1.0,71.527397,1.0,1,0,71.832443


In [45]:
X_new_std

,hpv_related,pack_years,uicc8_III-IV,oropharynx,cavum_oris,TLG
0,0.0,-1.101176,0.0,1,0,-0.179158
1,0.0,0.105775,0.0,0,0,-0.587333
2,0.0,0.705374,1.0,0,1,-0.192863
3,0.0,0.550384,0.0,0,0,-0.594926
4,0.0,1.233028,0.0,0,0,-0.540373
...,...,...,...,...,...,...
134,1.0,-1.101176,0.0,1,0,-0.488161
135,1.0,-1.101176,1.0,1,0,-0.099128
136,1.0,0.638406,0.0,1,0,-0.282017
137,1.0,2.049005,1.0,1,0,-0.253362


In [46]:
MAASTRO_new 

,hpv_related,pack_years,uicc8_III-IV,oropharynx,cavum_oris,TLG
0,1,0,0,1,0,263.611623
1,0,20,1,1,0,36.980700
2,0,6,1,1,0,74.636342
3,0,45,1,0,0,46.791979
4,1,59,0,1,0,107.637514
...,...,...,...,...,...,...
94,0,55,1,0,0,144.490782
95,0,174,1,0,0,69.214868
96,1,0,1,1,0,102.594274
97,1,0,0,1,0,103.229492


In [47]:
MAASTRO_new_std

,hpv_related,pack_years,uicc8_III-IV,oropharynx,cavum_oris,TLG
0,1,-1.101176,0,1,0,0.735160
1,0,-0.220344,1,1,0,-0.433005
2,0,-0.836927,1,1,0,-0.238909
3,0,0.880696,1,0,0,-0.382433
4,1,1.497278,0,1,0,-0.068806
...,...,...,...,...,...,...
94,0,1.321112,1,0,0,0.121154
95,0,6.562062,1,0,0,-0.266854
96,1,-1.101176,1,1,0,-0.094801
97,1,-1.101176,0,1,0,-0.091527


# Modelling 

### 1. CoxPHSurvivalAnalysis

#### Train

In [48]:
# Setting the y format for skf below  
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class()
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxPHSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxPHSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-13 17:20:39,423] A new study created in memory with name: no-name-18927288-354f-429f-83b7-85a73f109e97


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.6135458167330677
Fold 2 C-index: 0.5155038759689923
Fold 3 C-index: 0.7446808510638298


[I 2024-04-13 17:20:40,007] A new study created in memory with name: no-name-90d4e377-b179-471f-8f1b-bb5cbe239754


Fold 4 C-index: 0.6768060836501901
Fold 5 C-index: 0.6394849785407726
[I 2024-04-13 17:20:39,981] Trial 0 finished with value: 0.6380043211913705 and parameters: {}. Best is trial 0 with value: 0.6380043211913705.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.6380043211913705], datetime_start=datetime.datetime(2024, 4, 13, 17, 20, 39, 496844), datetime_complete=datetime.datetime(2024, 4, 13, 17, 20, 39, 981288), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.6380043211913705


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.23694732382610154
Fold 2 IBS: 0.2615158118717558
Fold 3 IBS: 0.17590965994359087
Fold 4 IBS: 0.28365588891632393
Fold 5 IBS: 0.19620686095559536
[I 2024-04-13 17:20:40,529] Trial 0 finished with value: 0.2308471091026735 and parameters: {}. Best is trial 0 with value: 0.2308471091026735.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.2308471091026735], datetime_start=datetime.datetime(2024, 4, 13, 17, 20, 40, 42810), datetime_complete=datetime.datetime(2024, 4, 13, 17, 20, 40, 529211), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.2308471091026735


In [49]:
# Setting a dictionary to save the train results 
train_cindex = {} 
train_ibs = {} 

# Saving the values to the dictionary 
train_cindex['CoxPH'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxPH'] = np.round(study_ibs.best_value, 3)

In [50]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.638
train_ibs:  0.231


#### Test

In [51]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [52]:
# Test on MAASTRO 
cph = CoxPHSurvivalAnalysis()

cph.fit(X_new_std, y)

# Save C-index 
c_index = cph.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print('Concordance index:', c_index)

# Save IBS 
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in cph.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print('IBS score:', ibs)

CoxPHSurvivalAnalysis()

Concordance index: 0.532
IBS score: 0.276


In [53]:
# Setting a dictionary to save the test results 
test_cindex = {} 
test_ibs = {} 

In [54]:
# Saving the values to the dictionary 
test_cindex['CoxPH'] = c_index
test_ibs['CoxPH'] = ibs

### 2. CoxnetSurvivalAnalysis

#### Train

In [55]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=0.0000001, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-13 17:20:40,888] A new study created in memory with name: no-name-604f00eb-3bbc-49d7-896b-9fa4e41eb42a


  0%|          | 0/1 [00:00<?, ?it/s]

[I 2024-04-13 17:20:41,257] A new study created in memory with name: no-name-e3a499b9-e314-4faf-a9ac-76b725d18c0d


Fold 1 C-index: 0.651394422310757
Fold 2 C-index: 0.5038759689922481
Fold 3 C-index: 0.5595744680851064
Fold 4 C-index: 0.5722433460076045
Fold 5 C-index: 0.6716738197424893
[I 2024-04-13 17:20:41,242] Trial 0 finished with value: 0.591752405027641 and parameters: {}. Best is trial 0 with value: 0.591752405027641.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.591752405027641], datetime_start=datetime.datetime(2024, 4, 13, 17, 20, 41, 3839), datetime_complete=datetime.datetime(2024, 4, 13, 17, 20, 41, 242133), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.591752405027641


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.2472470958068372
Fold 2 IBS: 0.2320398857267834
Fold 3 IBS: 0.22898186411767818
Fold 4 IBS: 0.2419747806817174
Fold 5 IBS: 0.22939558343033542
[I 2024-04-13 17:20:41,548] Trial 0 finished with value: 0.23592784195267033 and parameters: {}. Best is trial 0 with value: 0.23592784195267033.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.23592784195267033], datetime_start=datetime.datetime(2024, 4, 13, 17, 20, 41, 298898), datetime_complete=datetime.datetime(2024, 4, 13, 17, 20, 41, 548647), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.23592784195267033


In [56]:
train_cindex['CoxRidge'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxRidge'] = np.round(study_ibs.best_value, 3)

In [57]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.592
train_ibs:  0.236


#### Test

In [58]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [59]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 0.0000001
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_cindex : 0.532


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_ibs:  0.229


In [60]:
# Saving the values to the dictionary 
test_cindex['CoxRidge'] = c_index
test_ibs['CoxRidge'] = ibs

### 3. CoxnetSurvivalAnalysis - Lasso

#### Train

In [61]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=1, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-13 17:20:41,798] A new study created in memory with name: no-name-c8ff8980-f896-4996-8f73-ad11a41f5237


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.6095617529880478
Fold 2 C-index: 0.5155038759689923
Fold 3 C-index: 0.7468085106382979


[I 2024-04-13 17:20:42,443] A new study created in memory with name: no-name-7b64e401-238b-42af-b73e-7d9ec657d82c


Fold 4 C-index: 0.6692015209125475
Fold 5 C-index: 0.6394849785407726
[I 2024-04-13 17:20:42,432] Trial 0 finished with value: 0.6361121278097317 and parameters: {}. Best is trial 0 with value: 0.6361121278097317.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.6361121278097317], datetime_start=datetime.datetime(2024, 4, 13, 17, 20, 41, 882255), datetime_complete=datetime.datetime(2024, 4, 13, 17, 20, 42, 431962), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.6361121278097317


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.23871402060708521
Fold 2 IBS: 0.2596185745944766
Fold 3 IBS: 0.17748888379055622
Fold 4 IBS: 0.28227578624047067
Fold 5 IBS: 0.1962887806623178
[I 2024-04-13 17:20:43,149] Trial 0 finished with value: 0.2308772091789813 and parameters: {}. Best is trial 0 with value: 0.2308772091789813.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.2308772091789813], datetime_start=datetime.datetime(2024, 4, 13, 17, 20, 42, 518845), datetime_complete=datetime.datetime(2024, 4, 13, 17, 20, 43, 149409), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.2308772091789813


In [62]:
train_cindex['CoxLasso'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxLasso'] = np.round(study_ibs.best_value, 3)

In [63]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.636
train_ibs:  0.231


#### Test

In [64]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [65]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 1
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_cindex : 0.532


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_ibs:  0.273


In [66]:
# Saving the values to the dictionary 
test_cindex['CoxLasso'] = c_index
test_ibs['CoxLasso'] = ibs

### 4. CoxnetSurvivalAnalysis - ElasticNet

#### Train

In [67]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        l1_ratio = trial.suggest_float("l1_ratio", 0.0001, 1)
        
        # Create and fit survival model 
        model = model_class(l1_ratio=l1_ratio, 
                           fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-13 17:20:43,535] A new study created in memory with name: no-name-8e600e3d-7e40-4083-a1dd-c24fdc8890d6


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6095617529880478
Fold 2 C-index: 0.5155038759689923
Fold 3 C-index: 0.7468085106382979
Fold 4 C-index: 0.6692015209125475
Fold 5 C-index: 0.6437768240343348
[I 2024-04-13 17:20:44,214] Trial 0 finished with value: 0.6369704969084441 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.6369704969084441.
Fold 1 C-index: 0.6135458167330677
Fold 2 C-index: 0.5155038759689923
Fold 3 C-index: 0.7468085106382979
Fold 4 C-index: 0.6692015209125475
Fold 5 C-index: 0.6437768240343348
[I 2024-04-13 17:20:44,766] Trial 1 finished with value: 0.6377673096574481 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 1 with value: 0.6377673096574481.
Fold 1 C-index: 0.6135458167330677
Fold 2 C-index: 0.5155038759689923
Fold 3 C-index: 0.7468085106382979
Fold 4 C-index: 0.6692015209125475
Fold 5 C-index: 0.6437768240343348
[I 2024-04-13 17:20:45,377] Trial 2 finished with value: 0.6377673096574481 and parameters: {'l1_ratio': 0.22692876841884668}.

Fold 1 C-index: 0.6095617529880478
Fold 2 C-index: 0.5155038759689923
Fold 3 C-index: 0.7468085106382979
Fold 4 C-index: 0.6692015209125475
Fold 5 C-index: 0.6394849785407726
[I 2024-04-13 17:20:57,822] Trial 24 finished with value: 0.6361121278097317 and parameters: {'l1_ratio': 0.8723606880092448}. Best is trial 1 with value: 0.6377673096574481.
Fold 1 C-index: 0.6135458167330677
Fold 2 C-index: 0.5155038759689923
Fold 3 C-index: 0.7468085106382979
Fold 4 C-index: 0.6692015209125475
Fold 5 C-index: 0.6437768240343348
[I 2024-04-13 17:20:58,428] Trial 25 finished with value: 0.6377673096574481 and parameters: {'l1_ratio': 0.45903747271503514}. Best is trial 1 with value: 0.6377673096574481.
Fold 1 C-index: 0.6135458167330677
Fold 2 C-index: 0.5155038759689923
Fold 3 C-index: 0.7468085106382979
Fold 4 C-index: 0.6692015209125475
Fold 5 C-index: 0.6437768240343348
[I 2024-04-13 17:20:58,990] Trial 26 finished with value: 0.6377673096574481 and parameters: {'l1_ratio': 0.3527001258697712

Fold 1 C-index: 0.6135458167330677
Fold 2 C-index: 0.5155038759689923
Fold 3 C-index: 0.7468085106382979
Fold 4 C-index: 0.6692015209125475
Fold 5 C-index: 0.6437768240343348
[I 2024-04-13 17:21:12,819] Trial 48 finished with value: 0.6377673096574481 and parameters: {'l1_ratio': 0.2832727302146274}. Best is trial 1 with value: 0.6377673096574481.
Fold 1 C-index: 0.6135458167330677
Fold 2 C-index: 0.5155038759689923
Fold 3 C-index: 0.7468085106382979
Fold 4 C-index: 0.6692015209125475
Fold 5 C-index: 0.6437768240343348
[I 2024-04-13 17:21:13,306] Trial 49 finished with value: 0.6377673096574481 and parameters: {'l1_ratio': 0.4677828429786992}. Best is trial 1 with value: 0.6377673096574481.
Fold 1 C-index: 0.6135458167330677
Fold 2 C-index: 0.5155038759689923
Fold 3 C-index: 0.7468085106382979
Fold 4 C-index: 0.6692015209125475
Fold 5 C-index: 0.6437768240343348
[I 2024-04-13 17:21:13,833] Trial 50 finished with value: 0.6377673096574481 and parameters: {'l1_ratio': 0.556147470734502}.

...

  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 IBS: 0.23853069630534515
Fold 2 IBS: 0.25960504236444576
Fold 3 IBS: 0.1775955517345451
Fold 4 IBS: 0.28223154751356766
Fold 5 IBS: 0.19619977069315225
[I 2024-04-13 17:21:40,989] Trial 0 finished with value: 0.2308325217222112 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.2308325217222112.
Fold 1 IBS: 0.23827865105409132
Fold 2 IBS: 0.2596310519913947
Fold 3 IBS: 0.1775086410031393
Fold 4 IBS: 0.2822719285379554
Fold 5 IBS: 0.19605618940477867
[I 2024-04-13 17:21:41,655] Trial 1 finished with value: 0.23074929239827185 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 1 with value: 0.23074929239827185.
Fold 1 IBS: 0.23823724218535663
Fold 2 IBS: 0.2595408458083622
Fold 3 IBS: 0.17747364999597456
Fold 4 IBS: 0.28222578521701225
Fold 5 IBS: 0.19604077768278652
[I 2024-04-13 17:21:42,377] Trial 2 finished with value: 0.23070366017789845 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 2 with value: 0.2307036601778984

Fold 2 IBS: 0.25952948642752294
Fold 3 IBS: 0.177487100453673
Fold 4 IBS: 0.2822763910223109
Fold 5 IBS: 0.19609653824017242
[I 2024-04-13 17:21:54,215] Trial 25 finished with value: 0.23073350229300216 and parameters: {'l1_ratio': 0.3907387218790935}. Best is trial 11 with value: 0.23052662675257327.
Fold 1 IBS: 0.23840827161015396
Fold 2 IBS: 0.23355936404836403
Fold 3 IBS: 0.22532134089726802
Fold 4 IBS: 0.255508907355416
Fold 5 IBS: 0.22202737809913756
[I 2024-04-13 17:21:54,756] Trial 26 finished with value: 0.2349650524020679 and parameters: {'l1_ratio': 0.1300237528586279}. Best is trial 11 with value: 0.23052662675257327.
Fold 1 IBS: 0.24653102709732444
Fold 2 IBS: 0.23223780537994812
Fold 3 IBS: 0.2283578731101423
Fold 4 IBS: 0.24357460867753347
Fold 5 IBS: 0.22793004138059744
[I 2024-04-13 17:21:55,015] Trial 27 finished with value: 0.23572627112910913 and parameters: {'l1_ratio': 0.016560941611969082}. Best is trial 11 with value: 0.23052662675257327.
Fold 1 IBS: 0.238243184

Fold 1 IBS: 0.23827868339846203
Fold 2 IBS: 0.2596821730405745
Fold 3 IBS: 0.17756293006089313
Fold 4 IBS: 0.2822444645932325
Fold 5 IBS: 0.19610579077429535
[I 2024-04-13 17:22:07,608] Trial 50 finished with value: 0.2307748083734915 and parameters: {'l1_ratio': 0.4213768319549068}. Best is trial 11 with value: 0.23052662675257327.
Fold 1 IBS: 0.2382464870836125
Fold 2 IBS: 0.25967360063640943
Fold 3 IBS: 0.1774624054675995
Fold 4 IBS: 0.28227392623025543
Fold 5 IBS: 0.1960803495449635
[I 2024-04-13 17:22:08,588] Trial 51 finished with value: 0.23074735379256808 and parameters: {'l1_ratio': 0.34963290492960153}. Best is trial 11 with value: 0.23052662675257327.
Fold 1 IBS: 0.23827088067211144
Fold 2 IBS: 0.2596602575419221
Fold 3 IBS: 0.1774195775513413
Fold 4 IBS: 0.2823011257998262
Fold 5 IBS: 0.19603821377613406
[I 2024-04-13 17:22:09,290] Trial 52 finished with value: 0.23073801106826702 and parameters: {'l1_ratio': 0.24071415288105377}. Best is trial 11 with value: 0.230526626752

Fold 1 IBS: 0.2382502485563302
Fold 2 IBS: 0.2595485996036999
Fold 3 IBS: 0.17746018132291239
Fold 4 IBS: 0.2822062570392265
Fold 5 IBS: 0.1960634245460156
[I 2024-04-13 17:22:22,908] Trial 75 finished with value: 0.23070574221363693 and parameters: {'l1_ratio': 0.30001813220952506}. Best is trial 11 with value: 0.23052662675257327.
Fold 1 IBS: 0.23822428734846307
Fold 2 IBS: 0.25967158095524584
Fold 3 IBS: 0.17749156392799154
Fold 4 IBS: 0.2822924257486092
Fold 5 IBS: 0.1960563339011874
[I 2024-04-13 17:22:23,537] Trial 76 finished with value: 0.2307472383762994 and parameters: {'l1_ratio': 0.2909349888639582}. Best is trial 11 with value: 0.23052662675257327.
Fold 1 IBS: 0.2383174526414789
Fold 2 IBS: 0.2340216951026572
Fold 3 IBS: 0.22454648654604933
Fold 4 IBS: 0.2822564696089534
Fold 5 IBS: 0.22076476775707285
[I 2024-04-13 17:22:23,930] Trial 77 finished with value: 0.2399813743312423 and parameters: {'l1_ratio': 0.17366740919648238}. Best is trial 11 with value: 0.23052662675257

In [68]:
train_cindex['CoxElastic'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxElastic'] = np.round(study_ibs.best_value, 3)

In [69]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.638
train_ibs:  0.226


#### Test

In [70]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [71]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    return model_class(**best_params, fit_baseline_model=True)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.28621072101688444)

test_cindex : 0.532


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.18841437122569446)

test_ibs:  0.234


In [72]:
# Saving the values to the dictionary 
test_cindex['CoxElastic'] = c_index
test_ibs['CoxElastic'] = ibs

### 5. Random Survival Forest

#### Train

In [73]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None])
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics
        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score,
                            warm_start=warm_start,
                            max_depth=max_depth,
                            max_features=max_features,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_samples=max_samples, 
                            random_state=123)

        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])
            
            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(RandomSurvivalForest, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(RandomSurvivalForest, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-13 17:22:34,860] A new study created in memory with name: no-name-a861b805-48a7-405b-9456-93fee82b8251


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5976095617529881
Fold 2 C-index: 0.6143410852713178
Fold 3 C-index: 0.6212765957446809
Fold 4 C-index: 0.6806083650190115
Fold 5 C-index: 0.575107296137339
[I 2024-04-13 17:22:39,450] Trial 0 finished with value: 0.6177885807850675 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'max_samples': 0.7163467647263769, 'max_features': None, 'min_weight_fraction_leaf': 0.2192861223398122, 'warm_start': False}. Best is trial 0 with value: 0.6177885807850675.
Fold 1 C-index: 0.6095617529880478
Fold 2 C-index: 0.624031007751938
Fold 3 C-index: 0.6595744680851063
Fold 4 C-index: 0.7053231939163498
Fold 5 C-index: 0.6030042918454935
[I 2024-04-13 17:22:43,040] Trial 1 finished with value: 0.6402989429173871 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 5, 'min_samples_leaf': 4, 'max_depth': 11, 'n_estimators': 266, 'oob_score': False, 'max_samples': 0.7520097923745717, 'm

Fold 3 C-index: 0.7595744680851064
Fold 4 C-index: 0.7053231939163498
Fold 5 C-index: 0.6351931330472103
[I 2024-04-13 17:23:19,144] Trial 15 finished with value: 0.6757811535432272 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 2, 'min_samples_leaf': 2, 'max_depth': 7, 'n_estimators': 120, 'oob_score': True, 'max_samples': 0.9847369290686088, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.17574628605071868, 'warm_start': True}. Best is trial 14 with value: 0.7002369632470599.
Fold 1 C-index: 0.6235059760956175
Fold 2 C-index: 0.6686046511627907
Fold 3 C-index: 0.7212765957446808
Fold 4 C-index: 0.7243346007604563
Fold 5 C-index: 0.648068669527897
[I 2024-04-13 17:23:20,399] Trial 16 finished with value: 0.6771580986582885 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 7, 'min_samples_leaf': 17, 'max_depth': 7, 'n_estimators': 137, 'oob_score': True, 'max_samples': 0.962365055520953, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.1903778147805238

Fold 1 C-index: 0.6215139442231076
Fold 2 C-index: 0.6744186046511628
Fold 3 C-index: 0.774468085106383
Fold 4 C-index: 0.7395437262357415
Fold 5 C-index: 0.6952789699570815
[I 2024-04-13 17:23:33,709] Trial 30 finished with value: 0.7010446660346952 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 17, 'min_samples_leaf': 8, 'max_depth': 14, 'n_estimators': 169, 'oob_score': True, 'max_samples': 0.5345414788093061, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.03755841239520112, 'warm_start': True}. Best is trial 29 with value: 0.7116053815973882.
Fold 1 C-index: 0.6175298804780877
Fold 2 C-index: 0.6666666666666666
Fold 3 C-index: 0.7787234042553192
Fold 4 C-index: 0.7357414448669202
Fold 5 C-index: 0.7017167381974249
[I 2024-04-13 17:23:34,958] Trial 31 finished with value: 0.7000756268928836 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 17, 'min_samples_leaf': 8, 'max_depth': 14, 'n_estimators': 171, 'oob_score': True, 'max_samples': 0.5019192620095752

...

Fold 5 IBS: 0.2108108318582224
[I 2024-04-13 17:30:01,967] Trial 31 finished with value: 0.2206877222460263 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 9, 'min_samples_leaf': 9, 'max_depth': 3, 'n_estimators': 396, 'oob_score': True, 'max_samples': 0.2868712535426494, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.021231453689495153}. Best is trial 26 with value: 0.22015707552217512.
Fold 1 IBS: 0.24659836091991297
Fold 2 IBS: 0.23212875696426277
Fold 3 IBS: 0.23013831975076382
Fold 4 IBS: 0.24109773536375373
Fold 5 IBS: 0.22997643124737513
[I 2024-04-13 17:30:09,051] Trial 32 finished with value: 0.23598792084921366 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 10, 'min_samples_leaf': 9, 'max_depth': 3, 'n_estimators': 323, 'oob_score': True, 'max_samples': 0.15457524910822473, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.004932622259272009}. Best is trial 26 with value: 0.22015707552217512.
Fold 1 IBS: 0.23197141083929648
Fold 2 IBS: 0.22

Fold 1 IBS: 0.23103878287252402
Fold 2 IBS: 0.22628918231285983
Fold 3 IBS: 0.2133979486579942
Fold 4 IBS: 0.22430579562693312
Fold 5 IBS: 0.21153361401748674
[I 2024-04-13 17:32:32,978] Trial 47 finished with value: 0.22131306469755957 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 8, 'min_samples_leaf': 11, 'max_depth': 10, 'n_estimators': 475, 'oob_score': False, 'max_samples': 0.43491671167283275, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.07826676341691717}. Best is trial 26 with value: 0.22015707552217512.
Fold 1 IBS: 0.2461442212529148
Fold 2 IBS: 0.23226781881803804
Fold 3 IBS: 0.22981312479103597
Fold 4 IBS: 0.24139414747597257
Fold 5 IBS: 0.2304782442201671
[I 2024-04-13 17:32:44,432] Trial 48 finished with value: 0.23601951131162568 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 3, 'min_samples_leaf': 19, 'max_depth': 6, 'n_estimators': 447, 'oob_score': False, 'max_samples': 0.2751094610735481, 'max_features': 'auto', 'min_weight_fraction_

Fold 5 IBS: 0.23018854152461504
[I 2024-04-13 17:34:42,658] Trial 62 finished with value: 0.23591814060376043 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 5, 'min_samples_leaf': 5, 'max_depth': 10, 'n_estimators': 455, 'oob_score': True, 'max_samples': 0.4677262455345284, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.2827349986373208}. Best is trial 26 with value: 0.22015707552217512.
Fold 1 IBS: 0.24652405780792855
Fold 2 IBS: 0.2322152403880562
Fold 3 IBS: 0.22946718005561817
Fold 4 IBS: 0.2414651127800749
Fold 5 IBS: 0.23031058287471998
[I 2024-04-13 17:34:55,573] Trial 63 finished with value: 0.23599643478127957 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 8, 'min_samples_leaf': 14, 'max_depth': 9, 'n_estimators': 499, 'oob_score': True, 'max_samples': 0.36524236153301975, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.19147391488941964}. Best is trial 26 with value: 0.22015707552217512.
Fold 1 IBS: 0.23373423023692633
Fold 2 IBS: 0.2252

Fold 1 IBS: 0.24217247193439537
Fold 2 IBS: 0.2278793635566358
Fold 3 IBS: 0.20080078124083145
Fold 4 IBS: 0.22310960096298033
Fold 5 IBS: 0.2079672399605948
[I 2024-04-13 17:37:49,907] Trial 78 finished with value: 0.22038589153108754 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 4, 'min_samples_leaf': 2, 'max_depth': 13, 'n_estimators': 418, 'oob_score': False, 'max_samples': 0.7742508336660989, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.033857990321958055}. Best is trial 71 with value: 0.21814006458981802.
Fold 1 IBS: 0.2355555064593619
Fold 2 IBS: 0.22224942419207222
Fold 3 IBS: 0.20877935998830383
Fold 4 IBS: 0.22444536984656233
Fold 5 IBS: 0.2081228365598182
[I 2024-04-13 17:38:00,161] Trial 79 finished with value: 0.21983049940922367 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 2, 'min_samples_leaf': 4, 'max_depth': 14, 'n_estimators': 439, 'oob_score': False, 'max_samples': 0.8561349620215079, 'max_features': 'log2', 'min_weight_fraction_le

Fold 5 IBS: 0.2069805372099246
[I 2024-04-13 17:40:33,938] Trial 93 finished with value: 0.21836991119421248 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 3, 'min_samples_leaf': 3, 'max_depth': 15, 'n_estimators': 436, 'oob_score': False, 'max_samples': 0.7876680525303928, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.02818336790921719}. Best is trial 71 with value: 0.21814006458981802.
Fold 1 IBS: 0.2481959844323671
Fold 2 IBS: 0.23173657169109757
Fold 3 IBS: 0.20201510994966232
Fold 4 IBS: 0.22255626189257152
Fold 5 IBS: 0.21061654425734486
[I 2024-04-13 17:40:47,870] Trial 94 finished with value: 0.22302409444460863 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 5, 'min_samples_leaf': 3, 'max_depth': 16, 'n_estimators': 436, 'oob_score': False, 'max_samples': 0.8362507108284499, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.027504161893380293}. Best is trial 71 with value: 0.21814006458981802.
Fold 1 IBS: 0.23915650996574608
Fold 2 IBS: 0.223

In [74]:
train_cindex['Randomsurvivalforest'] = np.round(study_cindex.best_value, 3)
train_ibs['Randomsurvivalforest'] = np.round(study_ibs.best_value, 3)

In [75]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.79
train_ibs:  0.218


#### Test

In [76]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))
    
y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [77]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(RandomSurvivalForest, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex: ", c_index)

# Set the best model 
best_model_ibs = create_best_model(RandomSurvivalForest, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

RandomSurvivalForest(max_depth=20, max_features='auto', max_leaf_nodes=12,
                     max_samples=0.9937352645069003, min_samples_leaf=1,
                     min_samples_split=5,
                     min_weight_fraction_leaf=0.014733302673387358,
                     n_estimators=435, random_state=123, warm_start=True)

test_cindex:  0.603


RandomSurvivalForest(max_depth=13, max_features='log2', max_leaf_nodes=3,
                     max_samples=0.785271119394671, min_samples_leaf=4,
                     min_samples_split=2,
                     min_weight_fraction_leaf=8.074256882937306e-05,
                     n_estimators=444, random_state=123)

test_ibs:  0.215


In [78]:
# Saving the values to the dictionary 
test_cindex['Randomsurvivalforest'] = c_index
test_ibs['Randomsurvivalforest'] = ibs

### 6. ExtraSurvivalTrees

#### Train

In [79]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

In [80]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters 
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        warm_start = trial.suggest_categorical("warm_start", [True, False])
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics

        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score, 
                            max_features=max_features, 
                            warm_start=warm_start, 
                            max_samples=max_samples,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_depth=max_depth, 
                            random_state=123) 
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ExtraSurvivalTrees, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ExtraSurvivalTrees, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-13 17:41:48,733] A new study created in memory with name: no-name-c76c2f86-4499-4904-983f-5c30a19cf1c2


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5976095617529881
Fold 2 C-index: 0.6666666666666666
Fold 3 C-index: 0.7382978723404255
Fold 4 C-index: 0.7319391634980988
Fold 5 C-index: 0.6909871244635193
[I 2024-04-13 17:41:51,051] Trial 0 finished with value: 0.6851000777443397 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7641958651588321, 'min_weight_fraction_leaf': 0.09124586522674999}. Best is trial 0 with value: 0.6851000777443397.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-13 17:41:56,260] Trial 1 finished with value: 0.5 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 12, 'min_samples_leaf': 11, 'max_depth': 13, 'n_estimators': 425, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.4877764869966794, 'min_weight_fraction_leaf': 0.2468425488251531

Fold 4 C-index: 0.7243346007604563
Fold 5 C-index: 0.6158798283261803
[I 2024-04-13 17:43:04,406] Trial 15 finished with value: 0.6707959967448854 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 9, 'min_samples_leaf': 13, 'max_depth': 4, 'n_estimators': 258, 'oob_score': False, 'warm_start': True, 'max_features': 'sqrt', 'max_samples': 0.6880212408103346, 'min_weight_fraction_leaf': 0.07884420972256234}. Best is trial 12 with value: 0.6881534529891422.
Fold 1 C-index: 0.6055776892430279
Fold 2 C-index: 0.6589147286821705
Fold 3 C-index: 0.7085106382978723
Fold 4 C-index: 0.6977186311787072
Fold 5 C-index: 0.6094420600858369
[I 2024-04-13 17:43:08,090] Trial 16 finished with value: 0.656032749497523 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 17, 'min_samples_leaf': 5, 'max_depth': 20, 'n_estimators': 499, 'oob_score': False, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.8750346354626457, 'min_weight_fraction_leaf': 0.4093818399278544}. Best is

Fold 1 C-index: 0.6215139442231076
Fold 2 C-index: 0.6763565891472868
Fold 3 C-index: 0.7446808510638298
Fold 4 C-index: 0.7395437262357415
Fold 5 C-index: 0.628755364806867
[I 2024-04-13 17:44:01,014] Trial 30 finished with value: 0.6821700950953665 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 3, 'min_samples_leaf': 4, 'max_depth': 12, 'n_estimators': 355, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7262614295600427, 'min_weight_fraction_leaf': 0.020774312173092817}. Best is trial 23 with value: 0.6906324467750945.
Fold 1 C-index: 0.6095617529880478
Fold 2 C-index: 0.6550387596899225
Fold 3 C-index: 0.7404255319148936
Fold 4 C-index: 0.7186311787072244
Fold 5 C-index: 0.7017167381974249
[I 2024-04-13 17:44:03,983] Trial 31 finished with value: 0.6850747922995026 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 5, 'min_samples_leaf': 1, 'max_depth': 5, 'n_estimators': 400, 'oob_score': False, 'warm_start': True, 'max_features

...

Fold 1 IBS: 0.23738806550248728
Fold 2 IBS: 0.21993578889192347
Fold 3 IBS: 0.21089498091706652
Fold 4 IBS: 0.2272048730326933
Fold 5 IBS: 0.2132773803587104
[I 2024-04-13 17:53:43,167] Trial 30 finished with value: 0.2217402177405762 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 14, 'min_samples_leaf': 13, 'max_depth': 9, 'n_estimators': 206, 'oob_score': True, 'warm_start': False, 'max_features': 1, 'max_samples': 0.7311100810851326, 'min_weight_fraction_leaf': 0.16969817682654836}. Best is trial 20 with value: 0.21858490307862474.
Fold 1 IBS: 0.2371472983744441
Fold 2 IBS: 0.2192317816045922
Fold 3 IBS: 0.2066273024000569
Fold 4 IBS: 0.2223268600858535
Fold 5 IBS: 0.20805506169911736
[I 2024-04-13 17:53:54,940] Trial 31 finished with value: 0.21867766083281284 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 17, 'min_samples_leaf': 17, 'max_depth': 3, 'n_estimators': 259, 'oob_score': True, 'warm_start': False, 'max_features': 'sqrt', 'max_samples': 0.84794250

Fold 1 IBS: 0.24647150063770518
Fold 2 IBS: 0.2324005238674857
Fold 3 IBS: 0.2295165680261027
Fold 4 IBS: 0.2413350176086155
Fold 5 IBS: 0.23048981014618952
[I 2024-04-13 17:56:47,249] Trial 45 finished with value: 0.2360426840572197 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 18, 'min_samples_leaf': 18, 'max_depth': 8, 'n_estimators': 184, 'oob_score': True, 'warm_start': False, 'max_features': 'sqrt', 'max_samples': 0.42889080150309455, 'min_weight_fraction_leaf': 0.30395190085793466}. Best is trial 33 with value: 0.21855367964170153.
Fold 1 IBS: 0.23684554782082465
Fold 2 IBS: 0.21919333404974234
Fold 3 IBS: 0.2053013505980981
Fold 4 IBS: 0.22292477589597334
Fold 5 IBS: 0.20836678612341789
[I 2024-04-13 17:57:01,359] Trial 46 finished with value: 0.21852635889761127 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 20, 'min_samples_leaf': 19, 'max_depth': 6, 'n_estimators': 273, 'oob_score': True, 'warm_start': False, 'max_features': 'sqrt', 'max_samples': 0

Fold 1 IBS: 0.2427874690307322
Fold 2 IBS: 0.22051307489638503
Fold 3 IBS: 0.21103171261194614
Fold 4 IBS: 0.22387533549578809
Fold 5 IBS: 0.2083271491005745
[I 2024-04-13 18:01:15,710] Trial 60 finished with value: 0.22130694822708522 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 19, 'min_samples_leaf': 1, 'max_depth': 4, 'n_estimators': 74, 'oob_score': False, 'warm_start': False, 'max_features': 'log2', 'max_samples': 0.7017596527420207, 'min_weight_fraction_leaf': 0.09966382421080089}. Best is trial 46 with value: 0.21852635889761127.
Fold 1 IBS: 0.23644800149579903
Fold 2 IBS: 0.2195369287078321
Fold 3 IBS: 0.20653996690015303
Fold 4 IBS: 0.22236489358251474
Fold 5 IBS: 0.20806793260763023
[I 2024-04-13 18:01:39,699] Trial 61 finished with value: 0.21859154465878583 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 17, 'min_samples_leaf': 18, 'max_depth': 1, 'n_estimators': 357, 'oob_score': True, 'warm_start': False, 'max_features': 'auto', 'max_samples': 

Fold 1 IBS: 0.23757083460058645
Fold 2 IBS: 0.21936835681341144
Fold 3 IBS: 0.2050586023900099
Fold 4 IBS: 0.2231460133461373
Fold 5 IBS: 0.20844072705559955
[I 2024-04-13 18:05:10,423] Trial 75 finished with value: 0.21871690684114892 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 20, 'min_samples_leaf': 19, 'max_depth': 2, 'n_estimators': 220, 'oob_score': True, 'warm_start': False, 'max_features': 'auto', 'max_samples': 0.7803578114864529, 'min_weight_fraction_leaf': 0.017478919548903803}. Best is trial 73 with value: 0.21838002210423152.
Fold 1 IBS: 0.23640141704565878
Fold 2 IBS: 0.2197885256064894
Fold 3 IBS: 0.20318516882698215
Fold 4 IBS: 0.22544681939994865
Fold 5 IBS: 0.20897452087659646
[I 2024-04-13 18:05:23,826] Trial 76 finished with value: 0.21875929035113512 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 20, 'min_samples_leaf': 20, 'max_depth': 3, 'n_estimators': 293, 'oob_score': True, 'warm_start': False, 'max_features': 'auto', 'max_samples'

Fold 1 IBS: 0.237217800059635
Fold 2 IBS: 0.219560418627486
Fold 3 IBS: 0.2055584995526966
Fold 4 IBS: 0.22351950054338868
Fold 5 IBS: 0.20798078328178596
[I 2024-04-13 18:08:17,325] Trial 90 finished with value: 0.21876740041299841 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 19, 'min_samples_leaf': 11, 'max_depth': 5, 'n_estimators': 306, 'oob_score': False, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.6578960771293777, 'min_weight_fraction_leaf': 0.16477439734294264}. Best is trial 73 with value: 0.21838002210423152.
Fold 1 IBS: 0.236951540585596
Fold 2 IBS: 0.21950546902732596
Fold 3 IBS: 0.20504430850227562
Fold 4 IBS: 0.22315671121676062
Fold 5 IBS: 0.20782359042139878
[I 2024-04-13 18:08:28,126] Trial 91 finished with value: 0.21849632395067142 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 19, 'min_samples_leaf': 8, 'max_depth': 6, 'n_estimators': 282, 'oob_score': False, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.69

In [81]:
train_cindex['ExtraSurvivalTrees'] = np.round(study_cindex.best_value, 3)
train_ibs['ExtraSurvivalTrees'] = np.round(study_ibs.best_value, 3)

In [82]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.748
train_ibs:  0.218


#### Test

In [83]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [84]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ExtraSurvivalTrees, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ExtraSurvivalTrees, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ExtraSurvivalTrees(max_depth=11, max_leaf_nodes=17,
                   max_samples=0.8662742964997368, min_samples_leaf=1,
                   min_samples_split=19,
                   min_weight_fraction_leaf=0.0017689713740798967,
                   n_estimators=54, oob_score=True, random_state=123,
                   warm_start=True)

C-index score: 0.585


ExtraSurvivalTrees(max_depth=2, max_features='auto', max_leaf_nodes=20,
                   max_samples=0.7842377375661996, min_samples_leaf=19,
                   min_samples_split=19,
                   min_weight_fraction_leaf=0.04843949872109438,
                   n_estimators=291, oob_score=True, random_state=123)

IBS: 0.213


In [85]:
# Saving the values to the dictionary 
test_cindex['ExtraSurvivalTrees'] = c_index
test_ibs['ExtraSurvivalTrees'] = ibs

### 7. GradientBoostingSurvivalAnalysis

#### Train

In [86]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        criterion = trial.suggest_categorical('criterion', ['friedman_mse', 'squared_error'])
        ccp_alpha = trial.suggest_float("ccp_alpha", 0.0, 10)
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        min_impurity_decrease = trial.suggest_loguniform('min_impurity_decrease', 1e-7, 1e-1)
        validation_fraction = trial.suggest_float("validation_fraction", 0.0, 1.0)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            learning_rate=learning_rate,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            ccp_alpha=ccp_alpha, 
                            criterion=criterion,
                            min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            min_weight_fraction_leaf=min_weight_fraction_leaf,
                            max_depth=max_depth,
                            max_features=max_features,
                            max_leaf_nodes=max_leaf_nodes, 
                            min_impurity_decrease=min_impurity_decrease,
                            validation_fraction=validation_fraction, 
                            random_state=123)
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(GradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(GradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-13 18:10:08,563] A new study created in memory with name: no-name-2768d6b9-f594-4426-a7f9-39cd2b8d57b2


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-13 18:11:14,882] Trial 0 finished with value: 0.5 and parameters: {'subsample': 0.7268222670380755, 'learning_rate': 0.02932779416008757, 'dropout_rate': 0.3041663082077828, 'n_estimators': 276, 'criterion': 'friedman_mse', 'ccp_alpha': 9.807641983846155, 'min_weight_fraction_leaf': 0.34241486929243165, 'max_features': None, 'min_impurity_decrease': 2.4449249473284515e-05, 'validation_fraction': 0.7379954057320357, 'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 11, 'max_depth': 11}. Best is trial 0 with value: 0.5.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-13 18:11:49,189] Trial 1 finished with value: 0.5 and parameters: {'subsample': 0.6709608626961889, 'learning_rate': 0.08509374761370117, 'dropout_rate': 0.7520097923745717, 'n_estimators': 306, 'criterion': 'friedman_mse', 'ccp_alpha': 3.

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-13 18:29:25,626] Trial 13 finished with value: 0.5 and parameters: {'subsample': 0.8386796524426539, 'learning_rate': 0.046734492485875676, 'dropout_rate': 0.4821375662037144, 'n_estimators': 402, 'criterion': 'squared_error', 'ccp_alpha': 1.5696007313501796, 'min_weight_fraction_leaf': 0.18684147934268416, 'max_features': 'log2', 'min_impurity_decrease': 1.5044881127471587e-06, 'validation_fraction': 0.8166356053932342, 'min_samples_split': 16, 'max_leaf_nodes': 19, 'min_samples_leaf': 16, 'max_depth': 4}. Best is trial 12 with value: 0.6288581469840132.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-13 18:31:44,272] Trial 14 finished with value: 0.5 and parameters: {'subsample': 0.33235389014851724, 'learning_rate': 0.04522573411670834, 'dropout_rate': 0.2712811374536856, 'n_estimators': 405, 'criterion': 'squar

Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-13 18:52:35,958] Trial 25 finished with value: 0.5 and parameters: {'subsample': 0.7565257046780437, 'learning_rate': 0.01188344684416992, 'dropout_rate': 0.3433523077110169, 'n_estimators': 318, 'criterion': 'squared_error', 'ccp_alpha': 2.063559212730723, 'min_weight_fraction_leaf': 0.25401273903421573, 'max_features': 'auto', 'min_impurity_decrease': 5.773435946665558e-07, 'validation_fraction': 0.8062268184400477, 'min_samples_split': 16, 'max_leaf_nodes': 18, 'min_samples_leaf': 5, 'max_depth': 3}. Best is trial 12 with value: 0.6288581469840132.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-13 18:54:50,686] Trial 26 finished with value: 0.5 and parameters: {'subsample': 0.8922404482621683, 'learning_rate': 0.011585260292674536, 'dropout_rate': 0.2527000999648632, 'n_estimators': 446, 'criterion': 'friedman_mse', 'ccp_alpha': 0.11820935501930148, 'min_weight_fraction

...

Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-13 21:12:01,360] Trial 55 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.804828042321909, 'learning_rate': 0.023646082998228058, 'dropout_rate': 0.23926982708415356, 'n_estimators': 162, 'criterion': 'squared_error', 'ccp_alpha': 0.40074886285106287, 'min_weight_fraction_leaf': 0.2779066644542128, 'max_features': 'sqrt', 'min_impurity_decrease': 0.008146563872249941, 'validation_fraction': 0.8868940629916056, 'min_samples_split': 4, 'max_leaf_nodes': 16, 'min_samples_leaf': 19, 'max_depth': 4}. Best is trial 45 with value: 0.2332431097060344.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-13 21:12:24,168] Trial 56 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.6106355356441384, 'learning_rate': 0.0175381503002972, 'dropout_rate': 0.184039934

Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-13 21:14:32,850] Trial 66 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9043457557527063, 'learning_rate': 0.08720647345857328, 'dropout_rate': 0.27193939046019544, 'n_estimators': 154, 'criterion': 'squared_error', 'ccp_alpha': 1.4111498627316026, 'min_weight_fraction_leaf': 0.23517339076530247, 'max_features': 'sqrt', 'min_impurity_decrease': 6.086951842225704e-07, 'validation_fraction': 0.9127218528145804, 'min_samples_split': 4, 'max_leaf_nodes': 15, 'min_samples_leaf': 20, 'max_depth': 2}. Best is trial 63 with value: 0.23223812135106253.
Fold 1 IBS: 0.24724710044658996
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-13 21:14:34,922] Trial 67 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.8250695857763196, 'learning_rate': 0.0911086342832341, 'dropout_rate': 0.3723000

Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-13 21:17:11,735] Trial 77 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.7428517276581355, 'learning_rate': 0.07706989744762442, 'dropout_rate': 0.4476209058480567, 'n_estimators': 91, 'criterion': 'squared_error', 'ccp_alpha': 1.5888723787914292, 'min_weight_fraction_leaf': 0.16206442611838873, 'max_features': 0.1, 'min_impurity_decrease': 3.33671674086432e-07, 'validation_fraction': 0.9402673106586569, 'min_samples_split': 12, 'max_leaf_nodes': 13, 'min_samples_leaf': 17, 'max_depth': 9}. Best is trial 63 with value: 0.23223812135106253.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-13 21:17:24,642] Trial 78 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.7715268987373948, 'learning_rate': 0.08985503754923349, 'dropout_rate': 0.51064243866

Fold 4 IBS: 0.24197477145927118
Fold 5 IBS: 0.2293955930480925
[I 2024-04-13 21:18:56,516] Trial 88 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.6413430388178152, 'learning_rate': 0.0973528090376961, 'dropout_rate': 0.26912242502602596, 'n_estimators': 174, 'criterion': 'squared_error', 'ccp_alpha': 0.5807061908097002, 'min_weight_fraction_leaf': 0.16723537370276506, 'max_features': 0.1, 'min_impurity_decrease': 1.190634063232502e-06, 'validation_fraction': 0.8005899065401637, 'min_samples_split': 9, 'max_leaf_nodes': 11, 'min_samples_leaf': 17, 'max_depth': 4}. Best is trial 82 with value: 0.23205660370863482.
Fold 1 IBS: 0.24220714862378384
Fold 2 IBS: 0.22705284477038132
Fold 3 IBS: 0.22374120154251026
Fold 4 IBS: 0.23761138561741843
Fold 5 IBS: 0.22312495335199894
[I 2024-04-13 21:19:13,522] Trial 89 finished with value: 0.23074750678121858 and parameters: {'subsample': 0.38730242337646104, 'learning_rate': 0.08109800836387941, 'dropout_rate': 0.18122999

Fold 4 IBS: 0.23852342496708018
Fold 5 IBS: 0.22486084489133845
[I 2024-04-13 21:22:14,106] Trial 99 finished with value: 0.23142156664291713 and parameters: {'subsample': 0.40791967441342253, 'learning_rate': 0.07712333235501795, 'dropout_rate': 0.19859974199600133, 'n_estimators': 143, 'criterion': 'squared_error', 'ccp_alpha': 0.004426170739568851, 'min_weight_fraction_leaf': 0.16314282359709542, 'max_features': 'log2', 'min_impurity_decrease': 5.353566353972388e-07, 'validation_fraction': 0.1604355124738367, 'min_samples_split': 11, 'max_leaf_nodes': 15, 'min_samples_leaf': 14, 'max_depth': 4}. Best is trial 91 with value: 0.22930168836973525.


* Best trial for IBS: 
 FrozenTrial(number=91, state=TrialState.COMPLETE, values=[0.22930168836973525], datetime_start=datetime.datetime(2024, 4, 13, 21, 19, 46, 152887), datetime_complete=datetime.datetime(2024, 4, 13, 21, 20, 2, 75653), params={'subsample': 0.3646963250854607, 'learning_rate': 0.09255241886126767, 'dropout_rate': 0.123632

In [87]:
train_cindex['GradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['GradientBoosting'] = np.round(study_ibs.best_value, 3)

In [88]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.648
train_ibs:  0.229


#### Test

In [89]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [90]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(GradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(GradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

GradientBoostingSurvivalAnalysis(ccp_alpha=0.02647951541473475,
                                 criterion='squared_error',
                                 dropout_rate=0.14303111133639826,
                                 learning_rate=0.05317196534413593,
                                 max_depth=19, max_features=0.1,
                                 max_leaf_nodes=15,
                                 min_impurity_decrease=7.356647091617949e-06,
                                 min_samples_leaf=17, min_samples_split=19,
                                 min_weight_fraction_leaf=0.35066087197674667,
                                 n_estimators=78, random_state=123,
                                 subsample=0.9736544653699906,
                                 validation_fraction=0.6117331205297228)

C-index score: 0.639


GradientBoostingSurvivalAnalysis(ccp_alpha=0.03832008885345805,
                                 criterion='squared_error',
                                 dropout_rate=0.12363276406693596,
                                 learning_rate=0.09255241886126767, max_depth=4,
                                 max_leaf_nodes=15,
                                 min_impurity_decrease=6.12703415921239e-07,
                                 min_samples_leaf=16, min_samples_split=13,
                                 min_weight_fraction_leaf=0.14198218649980035,
                                 n_estimators=143, random_state=123,
                                 subsample=0.3646963250854607,
                                 validation_fraction=0.965806971606102)

IBS: 0.221


In [91]:
# Saving the values to the dictionary 
test_cindex['GradientBoosting'] = c_index
test_ibs['GradientBoosting'] = ibs

### 8. ComponentwiseGradientBoostingSurvivalAnalysis

#### Train

In [92]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

In [93]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            learning_rate=learning_rate,
                            random_state=123)
                
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective


# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best hyperparameters for C-index: \n", study_cindex.best_params)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best hyperparameters for IBS: \n", study_ibs.best_params)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-13 21:22:20,411] A new study created in memory with name: no-name-3ed9c695-297b-467d-b777-fea831c38bae


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5358565737051793
Fold 2 C-index: 0.4903100775193798
Fold 3 C-index: 0.5553191489361702
Fold 4 C-index: 0.5741444866920152
Fold 5 C-index: 0.6201716738197425
[I 2024-04-13 21:22:22,460] Trial 0 finished with value: 0.5551603921344974 and parameters: {'subsample': 0.7268222670380755, 'dropout_rate': 0.3575254014553415, 'n_estimators': 114, 'learning_rate': 0.05558016213920623}. Best is trial 0 with value: 0.5551603921344974.
Fold 1 C-index: 0.5358565737051793
Fold 2 C-index: 0.48643410852713176
Fold 3 C-index: 0.5553191489361702
Fold 4 C-index: 0.5741444866920152
Fold 5 C-index: 0.6244635193133047
[I 2024-04-13 21:22:39,600] Trial 1 finished with value: 0.5552435674347602 and parameters: {'subsample': 0.7475220728070068, 'dropout_rate': 0.4807958141120149, 'n_estimators': 491, 'learning_rate': 0.06879814411990147}. Best is trial 1 with value: 0.5552435674347602.
Fold 1 C-index: 0.599601593625498
Fold 2 C-index: 0.49806201550387597
Fold 3 C-index: 0.5595744680851064
Fold

Fold 1 C-index: 0.6175298804780877
Fold 2 C-index: 0.5310077519379846
Fold 3 C-index: 0.6148936170212767
Fold 4 C-index: 0.6463878326996197
Fold 5 C-index: 0.6609442060085837
[I 2024-04-13 21:24:49,086] Trial 19 finished with value: 0.6141526576291104 and parameters: {'subsample': 0.10013306718236009, 'dropout_rate': 0.7254192287154788, 'n_estimators': 117, 'learning_rate': 0.09614402133777997}. Best is trial 12 with value: 0.6175165352717875.
Fold 1 C-index: 0.6135458167330677
Fold 2 C-index: 0.5193798449612403
Fold 3 C-index: 0.5936170212765958
Fold 4 C-index: 0.5931558935361216
Fold 5 C-index: 0.648068669527897
[I 2024-04-13 21:25:02,406] Trial 20 finished with value: 0.5935534492069844 and parameters: {'subsample': 0.2630057481431337, 'dropout_rate': 0.18040218016888274, 'n_estimators': 423, 'learning_rate': 0.07788582119853761}. Best is trial 12 with value: 0.6175165352717875.
Fold 1 C-index: 0.6115537848605578
Fold 2 C-index: 0.5271317829457365
Fold 3 C-index: 0.5936170212765958


Fold 1 C-index: 0.6095617529880478
Fold 2 C-index: 0.5310077519379846
Fold 3 C-index: 0.5893617021276596
Fold 4 C-index: 0.6121673003802282
Fold 5 C-index: 0.6523605150214592
[I 2024-04-13 21:25:56,795] Trial 38 finished with value: 0.5988918044910758 and parameters: {'subsample': 0.1504722419513361, 'dropout_rate': 0.9099198027137476, 'n_estimators': 123, 'learning_rate': 0.09639188175417894}. Best is trial 12 with value: 0.6175165352717875.
Fold 1 C-index: 0.6055776892430279
Fold 2 C-index: 0.5096899224806202
Fold 3 C-index: 0.5765957446808511
Fold 4 C-index: 0.5817490494296578
Fold 5 C-index: 0.6394849785407726
[I 2024-04-13 21:26:00,593] Trial 39 finished with value: 0.5826194768749857 and parameters: {'subsample': 0.3126255482240327, 'dropout_rate': 0.29848665859072016, 'n_estimators': 198, 'learning_rate': 0.06892741183938003}. Best is trial 12 with value: 0.6175165352717875.
Fold 1 C-index: 0.6135458167330677
Fold 2 C-index: 0.49806201550387597
Fold 3 C-index: 0.5638297872340425

...

Fold 3 IBS: 0.2310576012464029
Fold 4 IBS: 0.27807484048509085
Fold 5 IBS: 0.21327756550890814
[I 2024-04-13 21:30:33,541] Trial 19 finished with value: 0.24675667086434197 and parameters: {'subsample': 0.6477795744201034, 'dropout_rate': 0.8173815053858949, 'n_estimators': 85, 'learning_rate': 0.03825176355689751}. Best is trial 17 with value: 0.23454251443542465.
Fold 1 IBS: 0.3380735899374104
Fold 2 IBS: 0.46733373445003945
Fold 3 IBS: 0.3220159248771053
Fold 4 IBS: 0.34521401041600086
Fold 5 IBS: 0.339374865845405
[I 2024-04-13 21:30:41,361] Trial 20 finished with value: 0.36240242510519216 and parameters: {'subsample': 0.8995742447068862, 'dropout_rate': 0.6343004869297953, 'n_estimators': 339, 'learning_rate': 0.0817175642960023}. Best is trial 17 with value: 0.23454251443542465.
Fold 1 IBS: 0.2447785911159304
Fold 2 IBS: 0.23613502747193257
Fold 3 IBS: 0.22167477293944274
Fold 4 IBS: 0.23857201410507312
Fold 5 IBS: 0.22438334443070537
[I 2024-04-13 21:30:42,234] Trial 21 finishe

Fold 3 IBS: 0.2440845459176938
Fold 4 IBS: 0.28704196088136535
Fold 5 IBS: 0.20403552363244765
[I 2024-04-13 21:31:16,131] Trial 38 finished with value: 0.2510600406992994 and parameters: {'subsample': 0.41067080060139016, 'dropout_rate': 0.13272164755980653, 'n_estimators': 201, 'learning_rate': 0.02608547591740891}. Best is trial 23 with value: 0.2327439633365859.
Fold 1 IBS: 0.24615375587585303
Fold 2 IBS: 0.24628523701425167
Fold 3 IBS: 0.2216904625741465
Fold 4 IBS: 0.27532206054080466
Fold 5 IBS: 0.21337163612042237
[I 2024-04-13 21:31:17,176] Trial 39 finished with value: 0.24056463042509563 and parameters: {'subsample': 0.6989144624450092, 'dropout_rate': 0.19258304600442933, 'n_estimators': 68, 'learning_rate': 0.0365566709162643}. Best is trial 23 with value: 0.2327439633365859.
Fold 1 IBS: 0.29575602982638
Fold 2 IBS: 0.36104264601421143
Fold 3 IBS: 0.2828990010999088
Fold 4 IBS: 0.2724769770883033
Fold 5 IBS: 0.288539279086917
[I 2024-04-13 21:31:28,082] Trial 40 finished w

Fold 4 IBS: 0.2912921867679323
Fold 5 IBS: 0.19946843946475085
[I 2024-04-13 21:31:53,989] Trial 57 finished with value: 0.24615503772671596 and parameters: {'subsample': 0.13165605060453595, 'dropout_rate': 0.6215636969984726, 'n_estimators': 282, 'learning_rate': 0.020570994628935912}. Best is trial 52 with value: 0.2323777864253694.
Fold 1 IBS: 0.24587951428848268
Fold 2 IBS: 0.25635516225894356
Fold 3 IBS: 0.22117244118528198
Fold 4 IBS: 0.23565881966280566
Fold 5 IBS: 0.21621898647169846
[I 2024-04-13 21:31:55,979] Trial 58 finished with value: 0.2350569847734425 and parameters: {'subsample': 0.9631039053260466, 'dropout_rate': 0.5584119442683281, 'n_estimators': 130, 'learning_rate': 0.017546400044930882}. Best is trial 52 with value: 0.2323777864253694.
Fold 1 IBS: 0.2886758087821922
Fold 2 IBS: 0.34737050785244566
Fold 3 IBS: 0.27633386064263016
Fold 4 IBS: 0.2841361207078176
Fold 5 IBS: 0.2690903893143384
[I 2024-04-13 21:31:58,234] Trial 59 finished with value: 0.293121337459

Fold 4 IBS: 0.23836292489739097
Fold 5 IBS: 0.21734062586140884
[I 2024-04-13 21:32:27,045] Trial 76 finished with value: 0.23401332790348445 and parameters: {'subsample': 0.9303178972102747, 'dropout_rate': 0.9996437200724101, 'n_estimators': 91, 'learning_rate': 0.020800331912285384}. Best is trial 65 with value: 0.23232679303698375.
Fold 1 IBS: 0.25110523525273676
Fold 2 IBS: 0.2739338359344842
Fold 3 IBS: 0.23026776767311077
Fold 4 IBS: 0.2373454159197639
Fold 5 IBS: 0.21574863218030504
[I 2024-04-13 21:32:34,894] Trial 77 finished with value: 0.24168017739208017 and parameters: {'subsample': 0.9980345772994771, 'dropout_rate': 0.9095068007760355, 'n_estimators': 337, 'learning_rate': 0.009675483731153595}. Best is trial 65 with value: 0.23232679303698375.
Fold 1 IBS: 0.24405377784762752
Fold 2 IBS: 0.24310270490752364
Fold 3 IBS: 0.21832495383057882
Fold 4 IBS: 0.24115697900725427
Fold 5 IBS: 0.2198396021449846
[I 2024-04-13 21:32:37,085] Trial 78 finished with value: 0.2332956035

Fold 5 IBS: 0.22780690516678317
[I 2024-04-13 21:32:57,352] Trial 95 finished with value: 0.23507044508915292 and parameters: {'subsample': 0.8966428006054885, 'dropout_rate': 0.7704366422806366, 'n_estimators': 30, 'learning_rate': 0.006842064022804307}. Best is trial 65 with value: 0.23232679303698375.
Fold 1 IBS: 0.2750994816533384
Fold 2 IBS: 0.3245226002709321
Fold 3 IBS: 0.2624205253911892
Fold 4 IBS: 0.253501837617046
Fold 5 IBS: 0.24486023772313253
[I 2024-04-13 21:32:58,897] Trial 96 finished with value: 0.2720809365311277 and parameters: {'subsample': 0.9529075957415207, 'dropout_rate': 0.9154736896330417, 'n_estimators': 110, 'learning_rate': 0.054487916689644846}. Best is trial 65 with value: 0.23232679303698375.
Fold 1 IBS: 0.2442539674574584
Fold 2 IBS: 0.24666942629822483
Fold 3 IBS: 0.21837620002874783
Fold 4 IBS: 0.23594320426076265
Fold 5 IBS: 0.21864610721665617
[I 2024-04-13 21:32:59,987] Trial 97 finished with value: 0.23277778105237 and parameters: {'subsample': 0

In [94]:
train_cindex['ComponentwiseGradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['ComponentwiseGradientBoosting'] = np.round(study_ibs.best_value, 3)

In [95]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.618
train_ibs:  0.232


#### Test

In [96]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [97]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.10707700162921632,
                                              learning_rate=0.09635955935176935,
                                              n_estimators=311,
                                              random_state=123,
                                              subsample=0.11211713718471546)

C-index score: 0.562


ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.8779048908531875,
                                              learning_rate=0.016037625960587432,
                                              n_estimators=74, random_state=123,
                                              subsample=0.9712842797437069)

IBS: 0.248


In [98]:
# Saving the values to the dictionary 
test_cindex['ComponentwiseGradientBoosting'] = c_index
test_ibs['ComponentwiseGradientBoosting'] = ibs

# Results

In [99]:
df_train_cindex = pd.DataFrame(train_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_train_cindex['rank'] = df_train_cindex['C-index'].rank(ascending=False)
df_train_cindex 

,C-index,rank
Randomsurvivalforest,0.790,1.0
ExtraSurvivalTrees,0.748,2.0
GradientBoosting,0.648,3.0
CoxPH,0.638,4.5
CoxElastic,0.638,4.5
CoxLasso,0.636,6.0
ComponentwiseGradientBoosting,0.618,7.0
CoxRidge,0.592,8.0


In [100]:
df_train_ibs = pd.DataFrame(train_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_train_ibs['rank'] = df_train_ibs['IBS'].rank(ascending=True)
df_train_ibs

,IBS,rank
Randomsurvivalforest,0.218,1.5
ExtraSurvivalTrees,0.218,1.5
CoxElastic,0.226,3.0
GradientBoosting,0.229,4.0
CoxPH,0.231,5.5
CoxLasso,0.231,5.5
ComponentwiseGradientBoosting,0.232,7.0
CoxRidge,0.236,8.0


In [101]:
df_test_cindex = pd.DataFrame(test_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_test_cindex['rank'] = df_test_cindex['C-index'].rank(ascending=False)
df_test_cindex 

,C-index,rank
GradientBoosting,0.639,1.0
Randomsurvivalforest,0.603,2.0
ExtraSurvivalTrees,0.585,3.0
ComponentwiseGradientBoosting,0.562,4.0
CoxPH,0.532,6.5
CoxRidge,0.532,6.5
CoxLasso,0.532,6.5
CoxElastic,0.532,6.5


In [102]:
df_test_ibs = pd.DataFrame(test_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_test_ibs['rank'] = df_test_ibs['IBS'].rank(ascending=True)
df_test_ibs 

,IBS,rank
ExtraSurvivalTrees,0.213,1.0
Randomsurvivalforest,0.215,2.0
GradientBoosting,0.221,3.0
CoxRidge,0.229,4.0
CoxElastic,0.234,5.0
ComponentwiseGradientBoosting,0.248,6.0
CoxLasso,0.273,7.0
CoxPH,0.276,8.0


In [103]:
# Renaming the column "index" to "model" 
df_train_cindex = df_train_cindex.reset_index().rename(columns={"index": "model"})
df_train_ibs = df_train_ibs.reset_index().rename(columns={"index": "model"})
df_test_cindex = df_test_cindex.reset_index().rename(columns={"index": "model"})
df_test_ibs = df_test_ibs.reset_index().rename(columns={"index": "model"})

# Save the files 
dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  
file_path = 'path_to_your_folder/'  

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']


dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  
file_path = '/Users/minjeongcheon/Desktop/results_thesis/d1/dfs/standard/rent/'  

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']

# Modify the file names to match the desired format
modified_file_names = ['d1_dfs_standard_rent_' + file_name for file_name in file_names]

# Loop through each DataFrame and save them with corresponding modified file names
for df, modified_file_name in zip(dfs, modified_file_names):
    file_path_name = file_path + modified_file_name  # Construct the full file path
    df.to_csv(file_path_name, index=False)  # Save the DataFrame to CSV file


In [104]:
from datetime import date
today = date.today()
print("Date: ", today)

Date:  2024-04-13
